In [2]:
import holidays
import numpy as np
import pandas as pd

import config
from config import features

UK_HOLIDAYS = pd.to_datetime(
    list(holidays.country_holidays('UK', years=range(2011, 2026)).keys())
)

raw_feature_name, target_feature_name = zip(*[
    ('Date', 'date'),
    ('Time', 'time'),
    ('Ozone', 'O3'),
    ('Nitric oxide', 'NO'),
    ('Nitrogen dioxide', 'NO2'),
    ('Carbon monoxide', 'CO'),
    ('Modelled Wind Direction', 'wind_dir'),
    ('Modelled Wind Speed', 'wind_speed'),
    ('Modelled Temperature', 'temp'),
    ('PM10 particulate matter (Hourly measured)', 'PM10'),
    ('PM2.5 particulate matter (Hourly measured)', 'PM2.5')
])

is_weekend = lambda date: date.dt.day_name().isin(['Saturday', 'Sunday'])
is_holiday = lambda date, holidays: date.isin(holidays)
is_off_day = lambda date, holidays: is_weekend(date) | is_holiday(date, holidays)


def get_day_category(date, holiday_dates):
    """Calculate day category: 0=workday, 1=weekend, 2=day before weekend"""
    next_date = date + pd.Timedelta(days=1)
    return pd.Categorical(np.select(
        [is_off_day(date, holiday_dates), is_off_day(next_date, holiday_dates)],
        [1, 2],
        default=0
    ))


def add_day_category(df, holiday_dates):
    category = get_day_category(df['date'], holiday_dates)
    return df.assign(day_category=lambda x: category)


def add_sin_cos(df, col_name):
    return df.assign(
        **{f'{col_name}_sin': lambda x, c=col_name: np.sin(np.radians(x[c])).round(4),
           f'{col_name}_cos': lambda x, c=col_name: np.cos(np.radians(x[c])).round(4)}
    )


def apply_min_max(df, exclude=None):
    x = df.select_dtypes(include='number')
    if exclude:
        x = x.drop(columns=exclude, errors='ignore')
    df[x.columns] = (2 * (x - x.min()) / (x.max() - x.min()) - 1).round(4)
    return df


def process(csv_year):
    print(f"Processing {csv_year}")
    return (
        pd.read_csv(
            f"{config.raw_csv}/{csv_year}.csv",
            na_values=['No data'],
            parse_dates=['Date'],
            skiprows=config.rows_to_skip,
            skipfooter=1,
            usecols=raw_feature_name,
            engine='python'
        )
        .rename(columns=dict(zip(raw_feature_name, target_feature_name)))
        .assign(time=lambda df: df['time'].str[:2].astype(int))
        .pipe(add_day_category, UK_HOLIDAYS)
        .drop(columns=['CO'])

    )

In [3]:
combined_raw = (
    pd.concat([process(year) for year in range(config.start_year, config.end_year + 1)])
    .sort_values(['date', 'time'])
)
# print(combined_raw.isna().sum())
# print(combined_raw[['O3', 'NO', 'NO2', 'PM10', 'PM2.5', 'wind_speed', 'temp', 'wind_dir']].isnull().any(axis=1).sum())

Processing 2012
Processing 2013
Processing 2014
Processing 2015
Processing 2016
Processing 2017
Processing 2018
Processing 2019
Processing 2020
Processing 2021
Processing 2022
Processing 2023
Processing 2024
Processing 2025


# COVID-19 šalinimas

In [4]:
combined_raw = combined_raw[~combined_raw['date'].dt.year.isin([2020, 2021, 2022])]
print(len(combined_raw))
print(combined_raw[['O3', 'NO', 'NO2', 'PM10', 'PM2.5', 'wind_speed', 'temp', 'wind_dir']].isnull().any(axis=1).sum())

96432
13875


# Neigiamų reikšmių skaičius ir šalinimas

In [5]:
print((combined_raw.select_dtypes(include='number') < 0).sum())
print('----')
# print(combined_raw[combined_raw['PM2.5'] < 0])
print('min max')
print(combined_raw.describe().loc[['min', 'max']])
print('HELLO')
print(combined_raw.isna().sum())
print('NYE')
cols = ['O3', 'NO', 'NO2', 'PM10', 'PM2.5']
mask = (combined_raw[cols] >= 0) | combined_raw[cols].isna()
combined_raw = combined_raw[mask.all(axis=1)]
print(combined_raw.isna().sum())
print(len(combined_raw))
print(combined_raw[['O3', 'NO', 'NO2', 'PM10', 'PM2.5', 'wind_speed', 'temp', 'wind_dir']].isnull().any(axis=1).sum())


time             0
O3              12
NO               5
NO2              0
PM10            21
PM2.5          185
wind_dir         0
wind_speed       0
temp          3732
dtype: int64
----
min max
                    date  time         O3         NO        NO2     PM10  \
min  2012-01-01 00:00:00   1.0   -0.93050   -0.37419    0.00000   -2.899   
max  2025-12-31 00:00:00  24.0  151.47363  872.83310  321.91104  187.900   

     PM2.5  wind_dir  wind_speed  temp  
min   -5.0       0.0         0.0 -10.5  
max  127.6     360.0        13.7  33.4  
HELLO
date               0
time               0
O3              2928
NO              2749
NO2             2749
PM10            5651
PM2.5           5221
wind_dir        2592
wind_speed      2592
temp            2592
day_category       0
dtype: int64
NYE
date               0
time               0
O3              2922
NO              2748
NO2             2748
PM10            5651
PM2.5           5221
wind_dir        2579
wind_speed      2579
temp    

# Wind dir sin cos, save to csv

In [6]:
combined_raw = (combined_raw
 .pipe(add_sin_cos, 'wind_dir')
 .drop(columns=['wind_dir']))

combined_raw.to_csv(config.unscaled_csv, index=False)


# Normalizavimas

In [7]:
from models import MinMaxScaler

cleaned = combined_raw.dropna()

EXCLUDE = ['wind_dir_sin', 'wind_dir_cos', 'day_category', 'time']
train_raw = cleaned[cleaned['date'].dt.year < 2025]
test_raw  = cleaned[cleaned['date'].dt.year == 2025]

scaler = MinMaxScaler()
scaler.fit(train_raw, exclude=EXCLUDE)

train_scaled = scaler.transform(train_raw)
test_scaled  = scaler.transform(test_raw)

scaled = pd.concat([train_scaled, test_scaled]).sort_values(['date', 'time'])
scaled.to_csv(config.scaled_csv, index=False)

In [21]:
print(f"Train (2012–2024) raw min/max PM2.5: {train_raw['PM2.5'].min():.2f} / {train_raw['PM2.5'].max():.2f}")
print(f"Test  (2025)      raw min/max PM2.5: {test_raw['PM2.5'].min():.2f} / {test_raw['PM2.5'].max():.2f}")


Train (2012–2024) raw min/max PM2.5: 0.00 / 127.60
Test  (2025)      raw min/max PM2.5: 0.00 / 81.00


# Block creation

In [9]:
from blocks import make_blocks
from KNNPredictor import KNNPredictor

INPUT_FEATURES = ['O3', 'NO', 'NO2', 'PM10', 'PM2.5', 'wind_speed', 'temp', 'wind_dir_sin', 'wind_dir_cos']
OUTPUT_FEATURES = ['PM2.5']
input_hours = [6, 7, 8]
forecast_hours = [9, 10, 11, 12, 13]

# Evaluation helpers

In [10]:
def compute_rmse(predictions, actuals):
    return np.sqrt(((predictions - actuals) ** 2).mean(axis=0))


def unscale_value(scaled_value, min_val, max_val):
    return (scaled_value + 1) * (max_val - min_val) / 2 + min_val

# Reporting helpers

In [11]:
def print_hourly_rmse(rmse_values, forecast_hours):
    for hour, r in zip(forecast_hours, rmse_values):
        print(f"  Valanda {hour}: RMSE {r:.4f}")
    print(f"Vidutinis RMSE: {np.mean(rmse_values):.4f}")


def compare_predictions(test_dates, predictions, actuals, forecast_hours, unscale_fn, n_samples=5):
    for i in range(min(n_samples, len(test_dates))):
        print(f"\n{test_dates[i].date()}:")
        for h_idx, hour in enumerate(forecast_hours):
            actual    = unscale_fn(actuals[i, h_idx])
            predicted = unscale_fn(predictions[i, h_idx])
            print(f"  Valanda {hour}: prognozė={predicted:.1f}, realiai={actual:.1f} µg/m³")

In [12]:
workdays = scaled[scaled['day_category'] == 0]
train_df  = workdays[workdays['date'].dt.year < 2025]
test_df   = workdays[workdays['date'].dt.year == 2025]

train_inputs, train_outputs, _          = make_blocks(train_df, INPUT_FEATURES, OUTPUT_FEATURES, input_hours, forecast_hours)
test_inputs,  test_outputs,  test_dates = make_blocks(test_df,  INPUT_FEATURES, OUTPUT_FEATURES, input_hours, forecast_hours)

knn = KNNPredictor(k=5)
knn.fit(train_inputs, train_outputs)
predictions = knn.predict(test_inputs)

rmse = compute_rmse(predictions, test_outputs)
print(f"Train blokų: {len(train_inputs)}, Test blokų: {len(test_inputs)}")
print_hourly_rmse(rmse, forecast_hours)

Train blokų: 1425, Test blokų: 131
  Valanda 9: RMSE 0.0973
  Valanda 10: RMSE 0.0905
  Valanda 11: RMSE 0.0995
  Valanda 12: RMSE 0.1060
  Valanda 13: RMSE 0.1069
Vidutinis RMSE: 0.1001


In [13]:
pm25_min = scaler.min_['PM2.5']
pm25_max = scaler.max_['PM2.5']

def unscale(val):
    return unscale_value(val, pm25_min, pm25_max)

print("PM2.5 RMSE (µg/m³):")
rmse_real = [
    np.sqrt(((unscale(predictions[:, h]) - unscale(test_outputs[:, h])) ** 2).mean())
    for h in range(len(forecast_hours))
]
print_hourly_rmse(rmse_real, forecast_hours)

PM2.5 RMSE (µg/m³):
  Valanda 9: RMSE 6.2107
  Valanda 10: RMSE 5.7736
  Valanda 11: RMSE 6.3502
  Valanda 12: RMSE 6.7656
  Valanda 13: RMSE 6.8188
Vidutinis RMSE: 6.3838


In [14]:
print("Pirmos 5 dienos:")
compare_predictions(test_dates, predictions, test_outputs, forecast_hours, unscale)

Pirmos 5 dienos:

2025-01-07:
  Valanda 9: prognozė=8.6, realiai=6.0 µg/m³
  Valanda 10: prognozė=9.4, realiai=5.0 µg/m³
  Valanda 11: prognozė=9.4, realiai=2.0 µg/m³
  Valanda 12: prognozė=11.0, realiai=4.0 µg/m³
  Valanda 13: prognozė=14.4, realiai=5.0 µg/m³

2025-01-08:
  Valanda 9: prognozė=14.8, realiai=8.0 µg/m³
  Valanda 10: prognozė=14.9, realiai=8.0 µg/m³
  Valanda 11: prognozė=18.0, realiai=11.0 µg/m³
  Valanda 12: prognozė=16.0, realiai=11.0 µg/m³
  Valanda 13: prognozė=19.5, realiai=10.0 µg/m³

2025-01-09:
  Valanda 9: prognozė=9.8, realiai=12.0 µg/m³
  Valanda 10: prognozė=9.1, realiai=15.0 µg/m³
  Valanda 11: prognozė=8.7, realiai=18.0 µg/m³
  Valanda 12: prognozė=9.0, realiai=19.0 µg/m³
  Valanda 13: prognozė=8.6, realiai=13.0 µg/m³

2025-01-13:
  Valanda 9: prognozė=10.2, realiai=15.0 µg/m³
  Valanda 10: prognozė=9.5, realiai=9.0 µg/m³
  Valanda 11: prognozė=9.8, realiai=11.0 µg/m³
  Valanda 12: prognozė=10.1, realiai=15.0 µg/m³
  Valanda 13: prognozė=9.4, realiai=15.0 

In [15]:
print(f"{'Val.':<6} {'Reali (vid.)':<16} {'KNN (vid.)':<14} {'Naive':<14} {'KNN RMSE':<12} {'Naive RMSE':<12} Geriau")
print("-" * 82)
for h_idx, hour in enumerate(forecast_hours):
    actual    = unscale(test_outputs[:, h_idx])
    knn_pred  = unscale(predictions[:, h_idx])
    naive_val = unscale(train_outputs[:, h_idx]).mean()

    knn_rmse   = np.sqrt(((knn_pred  - actual) ** 2).mean())
    naive_rmse = np.sqrt(((naive_val - actual) ** 2).mean())
    winner     = "KNN" if knn_rmse < naive_rmse else "Naive"

    print(f"{hour:<6} {actual.mean():<16.1f} {knn_pred.mean():<14.1f} {naive_val:<14.1f} {knn_rmse:<12.2f} {naive_rmse:<12.2f} {winner}")

Val.   Reali (vid.)     KNN (vid.)     Naive          KNN RMSE     Naive RMSE   Geriau
----------------------------------------------------------------------------------
9      13.0             13.2           18.2           6.21         12.33        KNN
10     12.6             13.3           18.1           5.77         11.82        KNN
11     11.8             13.5           17.9           6.35         11.51        KNN
12     11.6             13.3           17.6           6.77         10.81        KNN
13     11.0             13.3           17.5           6.82         10.38        KNN


In [16]:
print("Dienos su aukštu PM2.5 9 val. (>20 µg/m³):")
for i, date in enumerate(test_dates):
    if unscale(test_outputs[i, 0]) > 20:
        print(f"\n{date.date()}:")
        for h_idx, hour in enumerate(forecast_hours):
            actual    = unscale(test_outputs[i, h_idx])
            predicted = unscale(predictions[i, h_idx])
            print(f"  Valanda {hour}: prognozė={predicted:.1f}, realiai={actual:.1f} µg/m³")

Dienos su aukštu PM2.5 9 val. (>20 µg/m³):

2025-01-22:
  Valanda 9: prognozė=16.3, realiai=29.0 µg/m³
  Valanda 10: prognozė=16.4, realiai=27.0 µg/m³
  Valanda 11: prognozė=18.4, realiai=17.0 µg/m³
  Valanda 12: prognozė=17.6, realiai=24.0 µg/m³
  Valanda 13: prognozė=15.8, realiai=27.0 µg/m³

2025-02-06:
  Valanda 9: prognozė=14.4, realiai=21.0 µg/m³
  Valanda 10: prognozė=17.3, realiai=19.0 µg/m³
  Valanda 11: prognozė=14.5, realiai=15.0 µg/m³
  Valanda 12: prognozė=14.4, realiai=10.0 µg/m³
  Valanda 13: prognozė=15.1, realiai=5.0 µg/m³

2025-02-10:
  Valanda 9: prognozė=32.2, realiai=36.0 µg/m³
  Valanda 10: prognozė=34.7, realiai=37.0 µg/m³
  Valanda 11: prognozė=30.3, realiai=33.0 µg/m³
  Valanda 12: prognozė=28.8, realiai=35.0 µg/m³
  Valanda 13: prognozė=28.6, realiai=30.0 µg/m³

2025-02-11:
  Valanda 9: prognozė=20.8, realiai=29.0 µg/m³
  Valanda 10: prognozė=22.5, realiai=33.0 µg/m³
  Valanda 11: prognozė=25.6, realiai=38.0 µg/m³
  Valanda 12: prognozė=24.4, realiai=49.0 µg/m

In [17]:
date = '2025-03-11'
day_category = get_day_category(pd.Series([pd.Timestamp(date)]), UK_HOLIDAYS)[0]

hist_df = scaled[
    (scaled['date'] < date) &
    (scaled['day_category'] == day_category)
]
hist_inputs, hist_outputs, _ = make_blocks(hist_df, INPUT_FEATURES, OUTPUT_FEATURES, input_hours, forecast_hours)

day_input = (
    scaled[(scaled['date'] == date) & (scaled['time'].isin(input_hours))]
    .sort_values('time')[INPUT_FEATURES]
    .values.flatten()
)

prediction = KNNPredictor(k=5).fit(hist_inputs, hist_outputs).predict([day_input])[0]

actual = (
    scaled[(scaled['date'] == date) & (scaled['time'].isin(forecast_hours))]
    .sort_values('time')['PM2.5']
    .values
)

print(f"{date}:")
for h_idx, hour in enumerate(forecast_hours):
    print(f"  Valanda {hour}: prognozė={unscale(prediction[h_idx]):.1f}, realiai={unscale(actual[h_idx]):.1f} µg/m³")

2025-03-11:
  Valanda 9: prognozė=11.8, realiai=9.0 µg/m³
  Valanda 10: prognozė=9.9, realiai=8.0 µg/m³
  Valanda 11: prognozė=15.5, realiai=10.0 µg/m³
  Valanda 12: prognozė=8.3, realiai=7.0 µg/m³
  Valanda 13: prognozė=8.9, realiai=9.0 µg/m³


# Sliding window evaluation

In [18]:
def evaluate_sliding_windows(
    train_df,
    test_df,
    input_features,
    output_features,
    scaler,
    target_col='PM2.5',
    input_size=3,
    forecast_size=5,
    k=5,
    day_hours=24,
):
    """
    Evaluate KNN over all valid sliding windows in a day.

    For each start_hour in [1 .. day_hours - input_size - forecast_size + 1]:
        input_hours   = [start_hour, ..., start_hour + input_size - 1]
        forecast_hours = [start_hour + input_size, ..., start_hour + input_size + forecast_size - 1]

    Returns a DataFrame with one row per window containing:
        input_start, forecast_start, train_blocks, test_blocks,
        rmse_scaled_mean, rmse_real_mean,
        rmse_real_h<hour> for each forecast hour.
    """
    pm25_min = scaler.min_[target_col]
    pm25_max = scaler.max_[target_col]

    def unscale(v):
        return unscale_value(v, pm25_min, pm25_max)

    max_start = day_hours - input_size - forecast_size + 1
    rows = []

    for start in range(1, max_start + 1):
        in_hours  = list(range(start, start + input_size))
        out_hours = list(range(start + input_size, start + input_size + forecast_size))

        try:
            tr_in, tr_out, _         = make_blocks(train_df, input_features, output_features, in_hours, out_hours)
            te_in, te_out, te_dates  = make_blocks(test_df,  input_features, output_features, in_hours, out_hours)
        except ValueError:
            continue

        knn  = KNNPredictor(k=k)
        pred = knn.fit(tr_in, tr_out).predict(te_in)

        rmse_scaled = compute_rmse(pred, te_out)

        rmse_real = np.array([
            np.sqrt(((unscale(pred[:, h]) - unscale(te_out[:, h])) ** 2).mean())
            for h in range(forecast_size)
        ])

        row = {
            'input_start':    start,
            'input_hours':    in_hours,
            'forecast_start': start + input_size,
            'forecast_hours': out_hours,
            'train_blocks':   len(tr_in),
            'test_blocks':    len(te_in),
            'rmse_scaled_mean': rmse_scaled.mean(),
            'rmse_real_mean':   rmse_real.mean(),
        }
        for h_idx, hour in enumerate(out_hours):
            row[f'rmse_real_h{hour}'] = rmse_real[h_idx]

        rows.append(row)

    return pd.DataFrame(rows)


# --- Run ---
results = evaluate_sliding_windows(
    train_df       = workdays[workdays['date'].dt.year < 2025],
    test_df        = workdays[workdays['date'].dt.year == 2025],
    input_features = INPUT_FEATURES,
    output_features= OUTPUT_FEATURES,
    scaler         = scaler,
    input_size     = 3,
    forecast_size  = 5,
    k              = 5,
)

print(f"{'Input':<10} {'Forecast':<16} {'Train':>7} {'Test':>6} {'RMSE (µg/m³)':>13}")
print("-" * 58)
for _, r in results.iterrows():
    in_str  = f"{r['input_hours'][0]}-{r['input_hours'][-1]}"
    out_str = f"{r['forecast_hours'][0]}-{r['forecast_hours'][-1]}"
    print(f"{in_str:<10} {out_str:<16} {r['train_blocks']:>7.0f} {r['test_blocks']:>6.0f} {r['rmse_real_mean']:>13.2f}")

Input      Forecast           Train   Test  RMSE (µg/m³)
----------------------------------------------------------
1-3        4-8                 1606    145          6.28
2-4        5-9                 1608    147          6.69
3-5        6-10                1576    144          6.24
4-6        7-11                1527    145          6.43
5-7        8-12                1473    138          6.57
6-8        9-13                1425    131          6.38
7-9        10-14               1383    130          6.37
8-10       11-15               1344    130          6.09
9-11       12-16               1334    128          5.97
10-12      13-17               1339    131          6.05
11-13      14-18               1357    134          5.68
12-14      15-19               1397    137          5.67
13-15      16-20               1446    142          5.41
14-16      17-21               1500    151          5.85
15-17      18-22               1554    158          6.32
16-18      19-23             

In [19]:
all_days = scaled

results_all = evaluate_sliding_windows(
    train_df        = all_days[all_days['date'].dt.year < 2025],
    test_df         = all_days[all_days['date'].dt.year == 2025],
    input_features  = INPUT_FEATURES,
    output_features = OUTPUT_FEATURES,
    scaler          = scaler,
    input_size      = 3,
    forecast_size   = 5,
    k               = 5,
)

print(f"{'Input':<10} {'Forecast':<16} {'Train':>7} {'Test':>6} {'RMSE (µg/m³)':>13}")
print("-" * 58)
for _, r in results_all.iterrows():
    in_str  = f"{r['input_hours'][0]}-{r['input_hours'][-1]}"
    out_str = f"{r['forecast_hours'][0]}-{r['forecast_hours'][-1]}"
    print(f"{in_str:<10} {out_str:<16} {r['train_blocks']:>7.0f} {r['test_blocks']:>6.0f} {r['rmse_real_mean']:>13.2f}")

Input      Forecast           Train   Test  RMSE (µg/m³)
----------------------------------------------------------
1-3        4-8                 2936    270          6.03
2-4        5-9                 2952    275          6.21
3-5        6-10                2925    272          5.91
4-6        7-11                2871    272          5.97
5-7        8-12                2812    266          6.04
6-8        9-13                2772    259          5.88
7-9        10-14               2731    257          5.69
8-10       11-15               2688    260          5.66
9-11       12-16               2677    257          5.83
10-12      13-17               2684    262          5.64
11-13      14-18               2714    265          5.34
12-14      15-19               2761    269          5.44
13-15      16-20               2819    274          5.50
14-16      17-21               2874    283          6.01
15-17      18-22               2929    292          6.40
16-18      19-23             